# Fine-tune Didi (llama3.2:1b)

Runtime > Change runtime type > T4 GPU.

Upload `train.jsonl` and `test.jsonl` to the Colab Files panel before running the cells below.

In [1]:
# 1. Install (if this fails, copy the install cell from Unsloth's current Colab notebook)
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 46.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 53.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0━━━  0/13 [torchao]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━  6/13 [xformers]er]
    Found existing installation: transformers 5.22.1━━━━━━━━━━  6/13 [xformers]
    Uninstalling transformers-5.22.1:

In [31]:
# 2. Settings. The base model must match the Ollama baseline (llama3.2:1b).
# Check the model licence first before any use beyond this demo.
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
SYSTEM = (
    "You are Didi, a maternal and child health assistant for ASHA workers "
    "in rural India. You give accurate, safe health information about pregnancy, "
    "antenatal care, nutrition, and newborn care. "
    "You always respond in English. "
    "You never give dangerous medical advice. "
    "You always refer critical cases to a doctor or hospital immediately. "
    "If a food, herbal remedy, or home treatment is suggested as a replacement "
    "for a doctor's prescribed medicine or for urgent medical care, you always "
    "say this is not safe and advise seeing a doctor."
)
MAX_SEQ = 2048

In [32]:
# 3. Load the model in 4 bit and add LoRA adapters
from unsloth import FastLanguageModel

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


In [33]:
# 4. Load train.jsonl and turn each chat into one training text
from datasets import load_dataset

ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda ex: {"text": tok.apply_chat_template(
    ex["messages"], tokenize=False, add_generation_prompt=False)})
print(len(ds), "examples")
print(ds[0]["text"][:600])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/97 [00:00<?, ? examples/s]

97 examples
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Nov 2026

You are Didi, a maternal and child health assistant for ASHA workers in rural India. You give accurate, safe health information about pregnancy, antenatal care, nutrition, and newborn care. You always respond in English. You never give dangerous medical advice. You always refer critical cases to a doctor or hospital immediately. If a food, herbal remedy, or home treatment is suggested as a replacement for a doctor's prescribed medicine or for urgent medical care, you alwa


In [34]:
# 5. Train. If an argument name errors, copy the trainer cell from Unsloth's
# current notebook and keep the dataset cell above.
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=6,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to="none",
        seed=3407,
    ),
)
trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/97 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 97 | Num Epochs = 6 | Total steps = 78
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,3.605904
10,2.985306
15,2.426853
20,1.846332
25,1.379542
30,1.075444
35,0.970564
40,0.910994
45,0.856284
50,0.833094


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-78/tokenizer_config.json.


TrainOutput(global_step=78, training_loss=1.3621414166230421, metrics={'train_runtime': 84.1883, 'train_samples_per_second': 6.913, 'train_steps_per_second': 0.926, 'total_flos': 864606721056768.0, 'train_loss': 1.3621414166230421, 'epoch': 6.0})

In [16]:
# 6. Save the adapter to Drive so a session reset does not lose it
from google.colab import drive

drive.mount("/content/drive")
model.save_pretrained("/content/drive/MyDrive/didi-lora")
tok.save_pretrained("/content/drive/MyDrive/didi-lora")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/didi-lora/tokenizer_config.json',
 '/content/drive/MyDrive/didi-lora/chat_template.jinja',
 '/content/drive/MyDrive/didi-lora/tokenizer.json')

In [35]:
# 7. Answer the test prompts here and save answers_tuned.csv (the safe fallback)
import csv
import json

FastLanguageModel.for_inference(model)
rows = []
for line in open("test.jsonl", encoding="utf-8"):
    case = json.loads(line)
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": case["prompt"]}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    rows.append({"id": case["id"], "answer": answer.strip()})

with open("answers_tuned.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "answer"])
    writer.writeheader()
    writer.writerows(rows)
print("Saved answers_tuned.csv. Download it from the Files panel.")

Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Saved answers_tuned.csv. Download it from the Files panel.


In [ ]:
# 8. Export for Ollama. Look for the .gguf file in the folder it prints, then download it.
# Using q8_0 instead of q4_k_m: 4-bit quantization of the merged weights was
# washing out the LoRA adapter's small, precisely-learned adjustments on the
# cases that were already weakly learned (confirmed by comparing Colab's
# full-precision greedy output against the quantized local Ollama output on
# the same trained model - they diverged, and q4_k_m was worse). q8_0 keeps
# roughly double the precision at the cost of a larger file (~1.4-1.6GB).
model.save_pretrained_gguf("didi-ft", tok, quantization_method="q8_0")

In [37]:


import shutil
shutil.copytree("didi-ft_gguf", "/content/drive/MyDrive/didi-ft_gguf", dirs_exist_ok=True)
print("Copied to Google Drive — open drive.google.com and download didi-ft_gguf/ from there.")


Copied to Google Drive — open drive.google.com and download didi-ft_gguf/ from there.
